# 🛡️ FraudShield AI: Machine Learning & SHAP Explainability Pipeline
### Automated Insurance & Loan Claim Fraud Detection System
---
This notebook demonstrates the end-to-end Machine Learning and Explainable AI (XAI) pipeline powering **FraudShield AI**:
1. **Exploratory Data Analysis (EDA)** on the Kaggle Vehicle Insurance Fraud dataset (`fraud_oracle` schema)
2. **Feature Engineering & Preprocessing** with Scikit-Learn `ColumnTransformer` (One-Hot Encoding + Standard Scaling)
3. **Model Selection & Training** with **XGBoost Classifier** (`scale_pos_weight` for class imbalance)
4. **Performance Evaluation**: Accuracy, Precision, Recall, F1-Score, ROC-AUC, and Confusion Matrix
5. **Model Explainability with SHAP (SHapley Additive exPlanations)**: TreeExplainer, Global Summary Plots, and Per-Claim Local Explanations

## 1. Setup & Environment Imports

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import xgboost as xgb
import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

print("All libraries successfully imported!")

## 2. Load the Dataset
The dataset follows the standard **Kaggle Vehicle Insurance Claim Fraud Detection** benchmark (`fraud_oracle`).
It contains 33 features covering:
- **Demographics**: `Age`, `Sex`, `MaritalStatus`
- **Policy Details**: `PolicyType`, `VehicleCategory`, `VehiclePrice`, `Deductible`, `DriverRating`
- **Incident & Timing**: `Days_Policy_Accident`, `Days_Policy_Claim`, `Month`, `DayOfWeek`
- **Risk Indicators**: `PoliceReportFiled`, `WitnessPresent`, `Fault`, `PastNumberOfClaims`, `AddressChange_Claim`, `ClaimAmount`
- **Target**: `FraudFound_P` (1 = Fraudulent Claim, 0 = Legitimate Claim)

In [ ]:
data_path = os.path.join("..", "data", "raw", "vehicle_fraud_dataset.csv")
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("Fraud Distribution:")
print(df['FraudFound_P'].value_counts(normalize=True).round(3) * 100)
df.head()

## 3. Feature Definitions (Numerical vs Categorical)

In [ ]:
NUMERIC_FEATURES = [
    'Age', 'Deductible', 'DriverRating', 'ClaimAmount',
    'WeekOfMonth', 'WeekOfMonthClaimed', 'Year'
]

CATEGORICAL_FEATURES = [
    'Month', 'DayOfWeek', 'Make', 'AccidentArea', 'Sex',
    'MaritalStatus', 'Fault', 'PolicyType', 'VehicleCategory',
    'VehiclePrice', 'Days_Policy_Accident', 'Days_Policy_Claim',
    'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder',
    'PoliceReportFiled', 'WitnessPresent', 'AgentType',
    'NumberOfSupplements', 'AddressChange_Claim', 'NumberOfCars',
    'BasePolicy'
]

TARGET = 'FraudFound_P'
print(f"Defined {len(NUMERIC_FEATURES)} numeric and {len(CATEGORICAL_FEATURES)} categorical features.")

## 4. Preprocessing Pipeline (`ColumnTransformer`)
We use `StandardScaler` for numeric columns and `OneHotEncoder` with `handle_unknown='ignore'` for categorical columns.

In [ ]:
X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df[TARGET].values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), NUMERIC_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES)
    ]
)

X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

cat_encoder = preprocessor.named_transformers_['cat']
feature_names = NUMERIC_FEATURES + cat_encoder.get_feature_names_out(CATEGORICAL_FEATURES).tolist()
print(f"Transformed input feature space: {X_train.shape[1]} features.")

## 5. Model Selection: Why XGBoost (Extreme Gradient Boosting)?
### 🎯 Key Reasons for Choosing XGBoost:
1. **Tabular Performance**: Gradient-boosted decision trees consistently outperform neural networks on mixed numerical/categorical tabular datasets.
2. **Non-Linear Interactions**: In fraud detection, risks arise at complex intersections (e.g. *Incident within 7 days of policy creation + At-Fault policy holder + Absence of police report*). Decision trees map these interactions naturally.
3. **Loss-Function Imbalance Control**: Fraud datasets are naturally skewed. XGBoost's `scale_pos_weight = count(negative) / count(positive)` directly reweights the gradient step for the minority class.
4. **Regularization Against Overfitting**: Built-in tree depth limits (`max_depth`), column subsampling (`colsample_bytree`), and $L_1$/$L_2$ leaf weights penalize noise.
5. **Exact Polynomial SHAP Computation**: XGBoost models can be explained via Lundberg's `TreeExplainer` in $O(TLD^2)$ polynomial time rather than exponential sampling time, enabling real-time per-claim explainability in production APIs.

In [ ]:
pos_count = int(np.sum(y_train))
neg_count = len(y_train) - pos_count
scale_pos_weight = float(neg_count / max(pos_count, 1))
print(f"Calculated scale_pos_weight: {scale_pos_weight:.3f}")

model = xgb.XGBClassifier(
    n_estimators=160,
    max_depth=4,
    learning_rate=0.07,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train, y_train)
print("XGBoost Classifier successfully trained!")

## 6. Held-Out Evaluation on Unseen Test Data (20%)

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.48).astype(int)

print("=== HELD-OUT CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, target_names=['Legitimate (0)', 'Fraud (1)']))

print(f"Accuracy:  {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"Precision: {precision_score(y_test, y_pred)*100:.2f}%")
print(f"Recall:    {recall_score(y_test, y_pred)*100:.2f}%")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
cax = ax[0].matshow(cm, cmap=plt.cm.Blues, alpha=0.8)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax[0].text(x=j, y=i, s=cm[i, j], va='center', ha='center', size='xx-large', weight='bold')
ax[0].set_xticks([0, 1])
ax[0].set_yticks([0, 1])
ax[0].set_xticklabels(['Pred Legit', 'Pred Fraud'])
ax[0].set_yticklabels(['Actual Legit', 'Actual Fraud'])
ax[0].set_title('Confusion Matrix', fontsize=12, pad=15, weight='bold')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax[1].plot(fpr, tpr, color='#4f46e5', lw=2.5, label=f'ROC-AUC = {roc_auc_score(y_test, y_prob):.3f}')
ax[1].plot([0, 1], [0, 1], color='#94a3b8', linestyle='--')
ax[1].set_xlabel('False Positive Rate')
ax[1].set_ylabel('True Positive Rate (Recall)')
ax[1].set_title('Receiver Operating Characteristic (ROC)', fontsize=12, weight='bold')
ax[1].legend(loc='lower right')
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Model Explainability with SHAP (`TreeExplainer`)
SHAP values quantify each feature's contribution to the delta between the base prediction and the actual model output.

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
print(f"SHAP Explainer initialized. Output dimensions: {shap_values.shape}")

In [ ]:
# Global Feature Importance via Mean Absolute SHAP
mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
top_indices = np.argsort(mean_abs_shap)[::-1][:10]

plt.figure(figsize=(9, 5))
top_names = [feature_names[i].replace('cat__', '').replace('num__', '') for i in top_indices]
top_scores = [mean_abs_shap[i] for i in top_indices]

plt.barh(top_names[::-1], top_scores[::-1], color='#6366f1')
plt.xlabel('Mean Absolute SHAP Impact (|SHAP|)')
plt.title('Top 10 Global Features Driving Fraud Prediction', fontsize=12, weight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Local Inference: Explaining an Individual Claim (Why was it flagged?)
Here is how FraudShield AI explains an individual claim for insurance underwriters and reviewers.

In [ ]:
claim_idx = 0
claim_shap = shap_values[claim_idx]
claim_prob = y_prob[claim_idx]

results = []
for fname, val in zip(feature_names, claim_shap):
    results.append({
        'feature': fname.replace('cat__', '').replace('num__', ''),
        'shap_value': round(float(val), 4),
        'signal': 'Increases Fraud Risk (Alert)' if val > 0 else 'Reduces Fraud Risk (Safe)'
    })

res_df = pd.DataFrame(results).sort_values(by='shap_value', ascending=False)

print(f"Examining Claim #{claim_idx + 1}:")
print(f"Predicted Fraud Score: {claim_prob*100:.1f}% -> Tier: {'HIGH' if claim_prob >= 0.65 else 'MEDIUM' if claim_prob >= 0.35 else 'LOW'}")
print("\n--- Top 3 Fraud Drivers (Pushing Score UP) ---")
print(res_df[res_df['shap_value'] > 0].head(3))
print("\n--- Top 2 Mitigating Factors (Pulling Score DOWN) ---")
print(res_df[res_df['shap_value'] < 0].tail(2))